In [ ]:
# eda_pipeline.py
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.decomposition import PCA

# --- CONFIG ---
POSSIBLE_FEATURE_DIRS = ["features", os.path.join("dataset", "features")]
FEATURES_PATH = None
for d in POSSIBLE_FEATURE_DIRS:
    if os.path.isdir(d):
        FEATURES_PATH = d
        break
if FEATURES_PATH is None:
    raise FileNotFoundError(f"No features directory found. Checked: {POSSIBLE_FEATURE_DIRS}.")

OUT_DIR = "eda_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# --- LOAD CSVs ---
csv_files = sorted(glob.glob(os.path.join(FEATURES_PATH, "*_features.csv")) + glob.glob(os.path.join(FEATURES_PATH, "*.csv")))
if len(csv_files) == 0:
    raise FileNotFoundError(f"No CSV files found in {FEATURES_PATH}.")

dfs = []
for f in csv_files:
    try:
        df = pd.read_csv(f)
        df["__source_csv__"] = os.path.basename(f)
        dfs.append(df)
    except Exception as e:
        print("Failed to read", f, ":", e)
if len(dfs) == 0:
    raise RuntimeError("No CSV could be read. Check file formats.")

df_all = pd.concat(dfs, ignore_index=True, sort=False)
df_all.to_csv(os.path.join(OUT_DIR, "raw_concatenated_all.csv"), index=False)

# --- BASIC REPORT ---
basic_report = {
    "n_record_files": len(dfs),
    "n_rows_total": df_all.shape[0],
    "n_columns_total": df_all.shape[1]
}
pd.DataFrame([basic_report]).to_csv(os.path.join(OUT_DIR, "basic_report.csv"), index=False)

# --- DETECT LABEL COLUMN ---
label_col = None
for candidate in ["label", "Label", "stress_label", "target"]:
    if candidate in df_all.columns:
        label_col = candidate
        break
if label_col is None:
    for c in df_all.columns:
        if df_all[c].dtype == object:
            label_col = c
            break
if label_col is None:
    lastcol = df_all.columns[-1]
    if df_all[lastcol].nunique() <= 10:
        label_col = lastcol
if label_col is None:
    raise RuntimeError("Could not auto-detect a label column.")
print("Detected label column:", label_col)

# --- NUMERIC FEATURES & CLEANING ---
non_numeric = df_all.select_dtypes(exclude=[np.number]).columns.tolist()
if label_col in non_numeric:
    non_numeric.remove(label_col)
numeric_cols = [c for c in df_all.columns if c not in non_numeric and c != "__source_csv__"]

df_num = df_all[numeric_cols].copy()
df_num.drop(columns=df_num.columns[df_num.isna().all()], inplace=True)
df_num.drop(columns=df_num.columns[df_num.nunique() <= 1], inplace=True)

for c in df_num.columns:
    col = df_num[c].replace([np.inf, -np.inf], np.nan)
    if col.isna().any():
        med = col.median(skipna=True)
        col = col.fillna(med if not np.isnan(med) else 0.0)
    df_num[c] = col

if label_col not in df_num.columns:
    df_num[label_col] = df_all[label_col].values

# --- FIX: CLIP EXTREME VALUES (1st–99th percentile per column) ---
for c in df_num.drop(columns=[label_col], errors="ignore").columns:
    lower, upper = np.nanpercentile(df_num[c], [1, 99])
    df_num[c] = np.clip(df_num[c], lower, upper)

df_num.to_csv(os.path.join(OUT_DIR, "concatenated_cleaned_features.csv"), index=False)

# --- SUMMARY & MISSING REPORT ---
df_num.describe().transpose().to_csv(os.path.join(OUT_DIR, "feature_summary_stats.csv"))
missing_report = df_all.isna().sum().rename("n_missing").to_frame()
missing_report["pct_missing"] = missing_report["n_missing"] / len(df_all) * 100.0
missing_report.to_csv(os.path.join(OUT_DIR, "missing_report.csv"))

# --- CORRELATION MATRIX ---
corr = df_num.drop(columns=[label_col], errors='ignore').corr(method='pearson')
corr.to_csv(os.path.join(OUT_DIR, "correlation_matrix.csv"))
plt.figure(figsize=(10,8))
plt.imshow(corr.values, aspect='auto', interpolation='nearest')
plt.colorbar()
plt.title("Feature Correlation (Pearson)")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90, fontsize=8)
plt.yticks(range(len(corr.columns)), corr.columns, fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "correlation_heatmap.png"))
plt.close()

# --- PREPARE X, y ---
X_df = df_num.drop(columns=[label_col], errors='ignore').copy()
X_df = X_df.apply(pd.to_numeric, errors="coerce").fillna(0.0)

# FIX: Use RobustScaler to reduce influence of outliers
scaler = RobustScaler()
X = scaler.fit_transform(X_df).astype(np.float64)

feature_names = X_df.columns.tolist()
y_raw = df_num[label_col].values
le = LabelEncoder()
y = le.fit_transform(y_raw)
pd.DataFrame({"label": le.classes_, "encoded": range(len(le.classes_))}).to_csv(os.path.join(OUT_DIR, "label_mapping.csv"), index=False)

# --- TRAIN/TEST SPLIT ---
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.20, random_state=42)

# --- RANDOM FOREST FEATURE IMPORTANCE ---
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
clf_report = classification_report(y_test, y_pred, output_dict=True)
pd.DataFrame(clf_report).transpose().to_csv(os.path.join(OUT_DIR, "classification_report_rf.csv"))
importances = rf.feature_importances_
feat_imp_df = pd.DataFrame({"feature": feature_names, "importance": importances}).sort_values("importance", ascending=False)
feat_imp_df.to_csv(os.path.join(OUT_DIR, "rf_feature_importances.csv"), index=False)

topk = min(20, len(feat_imp_df))
plt.figure(figsize=(8,6))
top_feats = feat_imp_df.head(topk)
plt.barh(range(topk), top_feats['importance'].values[::-1])
plt.yticks(range(topk), top_feats['feature'].values[::-1])
plt.xlabel("Importance")
plt.title("Top feature importances (Random Forest)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "rf_top_importances.png"))
plt.close()

# --- MUTUAL INFORMATION ---
mi = mutual_info_classif(X, y, random_state=42)
mi_df = pd.DataFrame({"feature": feature_names, "mutual_info": mi}).sort_values("mutual_info", ascending=False)
mi_df.to_csv(os.path.join(OUT_DIR, "mutual_information.csv"), index=False)

# --- PCA ---
pca = PCA(n_components=min(20, X.shape[1]))
pca.fit(X)
explained = pca.explained_variance_ratio_
pd.DataFrame({"component": [f"PC{i+1}" for i in range(len(explained))],
              "explained_variance": explained}).to_csv(os.path.join(OUT_DIR, "pca_explained_variance.csv"), index=False)

plt.figure(figsize=(6,4))
plt.plot(np.cumsum(explained), marker='o')
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA cumulative explained variance")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "pca_cumulative_variance.png"))
plt.close()

# --- PAIRWISE SCATTER OF TOP FEATURES ---
top_features = feat_imp_df['feature'].values[:6].tolist()
if len(top_features) >= 2:
    fig, axs = plt.subplots(len(top_features), len(top_features), figsize=(12,12))
    for i, fi in enumerate(top_features):
        for j, fj in enumerate(top_features):
            if i == j:
                axs[i,j].text(0.5, 0.5, fi, ha='center', va='center')
                axs[i,j].set_xticks([]); axs[i,j].set_yticks([])
            else:
                axs[i,j].scatter(df_num[fi].values, df_num[fj].values, s=8)
            if i == len(top_features)-1:
                axs[i,j].set_xlabel(fj, fontsize=8)
            if j == 0:
                axs[i,j].set_ylabel(fi, fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "pairwise_top_features.png"))
    plt.close()

print("✅ EDA finished. Files saved to:", OUT_DIR)


Detected label column: label


c:\Users\prana\anaconda3\envs\tds\lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: overflow encountered in square
  sqr = _ensure_numeric((avg - values) ** 2)


✅ EDA finished. Files saved to: eda_outputs
